In [3]:
import pandas as pd
from pathlib import Path
import os
from dotenv import dotenv_values

In [4]:
data_path = Path(os.path.abspath('')).resolve().parent / "data.csv"
env_path = Path(os.path.abspath('')).resolve().parent / "secrets.env"

df = pd.read_csv(data_path)

env_vars = dotenv_values(env_path)
# Filter out bad coordinates

df = df.dropna(subset = ['longitude', 'latitude'])
df = df[(df['longitude'] != 0) & (df['latitude'] != 0)]

df['street_name'] = df['street_name'].apply(lambda x : x.title())

df['address'] = df.apply(lambda x : str(x['house_number']) + ' ' + x['street_name'] ,axis = 1)
print(df.shape)
df.head()

(970, 26)


,inspection_type,job_ticket_or_work_order_id,job_id,job_progress,bbl,boro_code,block,lot,house_number,street_name,...,location,community_board,council_district,census_tract,bin,nta,approved_date,x_coord,y_coord,address
0,BAIT,3056322,PC8487803,5,3.019150e+09,3,1915,61,284,Clinton Avenue,...,"{'latitude': '40.689887042515', 'longitude': '...",2.0,35.0,195.0,3055028.0,Clinton Hill,NaN,NaN,NaN,284 Clinton Avenue
1,Initial,14208272,PC8635827,1,3.030980e+09,3,3098,19,172,Seigel Street,...,"{'latitude': '40.704790525696', 'longitude': '...",1.0,34.0,489.0,3071470.0,East Williamsburg,2025-12-09T14:52:51.000,NaN,NaN,172 Seigel Street
2,BAIT,3056363,PC8548734,7,3.051220e+09,3,5122,10,116,East 19 Street,...,"{'latitude': '40.646503808371', 'longitude': '...",14.0,40.0,512.0,3117591.0,Flatbush,NaN,994915.0,174820.0,116 East 19 Street
3,BAIT,3056271,PC8612711,6,1.021080e+09,1,2108,52,469,West 157 Street,...,"{'latitude': '40.832524213959', 'longitude': '...",12.0,10.0,239.0,1062508.0,Washington Heights (South),2025-12-11T11:30:29.000,1000516.0,242596.0,469 West 157 Street
4,BAIT,3056414,PC8609532,4,2.033570e+09,2,3357,248,415,East 204 Street,...,"{'latitude': '40.870789389041', 'longitude': '...",7.0,11.0,425.0,2018686.0,Norwood,2025-12-09T09:20:49.000,NaN,NaN,415 East 204 Street


In [5]:
test_row = df.iloc[0]

for row in df.head(10).iterrows():
    latitude = row[1]['latitude']
    longitude = row[1]['longitude']
    address = row[1]['address']
    print(row[0],latitude, longitude, address)

0 40.689887042515 -73.968177844567 284 Clinton Avenue
1 40.704790525696 -73.939378259695 172 Seigel Street
2 40.646503808371 -73.961568071532 116 East   19 Street
3 40.832524213959 -73.94122003788 469 West  157 Street
4 40.870789389041 -73.87651098883 415 East 204 Street
5 40.767160410349 -73.897685854609 71-11 Astoria Boulevard N
6 40.790615419424 -73.948436619708 122 East 103 Street
7 40.71526031998 -73.984420123573 460 Grand Street
8 40.682307685049 -73.964644481108 554 Washington Avenue
9 40.682307685049 -73.964644481108 554 Washington Avenue


In [6]:
print(results)

NameError: name 'results' is not defined

In [ ]:
import requests
import json

# Los tacos lat/lon

# lat = 40.7065340451609
# lon = -74.0116639196217

# Bk restaurant example
lat = 40.689425
lon = -73.9723664

# Demo
                # "latitude": 37.7937,
                # "longitude": -122.3965

# Replace with your actual API Key
API_KEY = env_vars['GOOGLE_PLACES_API_KEY']

# The endpoint for the Nearby Search (New) API
url = "https://places.googleapis.com/v1/places:searchNearby"

# Define the request body as a Python dictionary
# This example searches for restaurants within a 500-meter radius
# around a specific latitude and longitude.
request_body = {
    "includedTypes": ["restaurant", "meal_delivery", "meal_takeaway", "bakery", "bar", "cafe"],
    "maxResultCount": 20,
    "locationRestriction": {
        "circle": {
            "center": {
                "latitude": lat,
                "longitude": lon
            },
            "radius": 10.0
        }
    }
}

# Define the headers for the request
# X-Goog-Api-Key is used for authentication.
# X-Goog-FieldMask specifies which fields to return in the response.
# It's good practice to only request the fields you need to save on processing time and billing.
headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.location,places.rating"
}

try:
    # Make the POST request
    # The 'json' parameter in requests.post automatically serializes the dictionary to JSON
    # and sets the 'Content-Type' header to 'application/json'.
    response = requests.post(url, headers=headers, json=request_body)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        response_data = response.json()
        print("Nearby Search Results:")
        if response_data and "places" in response_data:
            for place in response_data["places"]:
                display_name = place.get("displayName", {}).get("text", "N/A")
                formatted_address = place.get("formattedAddress", "N/A")
                location = place.get("location", {})
                rating = place.get("rating", "N/A")
                print(f"  Name: {display_name}")
                print(f"  Address: {formatted_address}")
                print(f"  Location: Latitude {location.get('latitude')}, Longitude {location.get('longitude')}")
                print(f"  Rating: {rating}")
                print("-" * 20)
        else:
            print("No places found or unexpected response format.")
    else:
        print(f"Error: Request failed with status code {response.status_code}")
        print(f"Response: {response.text}")

except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")



Nearby Search Results:
  Name: Miss Ada
  Address: 184 DeKalb Ave, Brooklyn, NY 11205, USA
  Location: Latitude 40.689425, Longitude -73.9723664
  Rating: 4.6
--------------------


In [7]:
# pd.DataFrame(response_data['places'])
request_body['locationRestriction']['circle']['center']['latitude']
request_body['locationRestriction']['circle']['center']['longitude']

NameError: name 'request_body' is not defined

In [9]:
import requests
import time
# Replace with your actual API Key
API_KEY = env_vars['GOOGLE_PLACES_API_KEY']

# The endpoint for the Nearby Search (New) API
url = "https://places.googleapis.com/v1/places:searchNearby"

# Define the request body as a Python dictionary
# This example searches for restaurants within a 500-meter radius
# around a specific latitude and longitude.
request_body = {
    "includedTypes": ["restaurant", "meal_delivery", "meal_takeaway", "bakery", "bar", "cafe"],
    "maxResultCount": 20,
    "locationRestriction": {
        "circle": {
            "center": {
                "latitude": 0,
                "longitude": 0
            },
            "radius": 25.0
        }
    }
}

headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": API_KEY,
    "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.location,places.rating"
}


requested_places = pd.DataFrame()


for row in df.head(100).iterrows():

    temp_df = pd.DataFrame()

    latitude = row[1]['latitude']
    longitude = row[1]['longitude']
    address = row[1]['address']
    
    try:
        # Make the POST request
        # The 'json' parameter in requests.post automatically serializes the dictionary to JSON
        # and sets the 'Content-Type' header to 'application/json'.

        # Reset for every iteration
        request_body['locationRestriction']['circle']['center']['latitude'] = latitude
        request_body['locationRestriction']['circle']['center']['longitude'] = longitude

        # Make request
        response = requests.post(url, headers=headers, json=request_body)
        time.sleep(0.25)
        print(row[0])
        # Check if the request was successful (status code 200)
        if response.status_code == 200:
            response_data = response.json()
            print("Nearby Search Results:")
            if response_data and "places" in response_data:
                for place in response_data["places"]:
                    display_name = place.get("displayName", {}).get("text", "N/A")
                    formatted_address = place.get("formattedAddress", "N/A")
                    location = place.get("location", {})
                    rating = place.get("rating", "N/A")
                    # print(f"  Name: {display_name}")
                    # print(f"  Address: {formatted_address}")
                    # print(f"  Location: Latitude {location.get('latitude')}, Longitude {location.get('longitude')}")
                    # print(f"  Rating: {rating}")
                    # print("-" * 20)
            else:
                print(response)
                print("No places found or unexpected response format.")
        else:
            print(f"Error: Request failed with status code {response.status_code}")
            print(f"Response: {response.text}")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
    
    if response_data:
        temp_df = pd.DataFrame(response_data['places'])
        temp_df['latitude'] = latitude
        temp_df['longitude'] = longitude
        temp_df['address'] = address
        
        requested_places = pd.concat([requested_places, temp_df])
    else:
        print("Nothing returned!")
        print(response_data)

0
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
1
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
2
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
3
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
4
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
5
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
6
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
7
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
8
Nearby Search Results:
<Response [200]>
No places found or unexpected response format.
Nothing returned!
{}
9
Nearby S

In [10]:
requested_places.to_csv("places_v3_fast_pulls.csv", index = False)
requested_places

,formattedAddress,location,rating,displayName,latitude,longitude,address
0,"554 W 181st St, New York, NY 10033, USA","{'latitude': 40.8488258, 'longitude': -73.9328...",2.7,"{'text': 'Caridad', 'languageCode': 'en'}",40.848947,-73.932874,554 West 181 Street
1,"554 W 181st St, New York, NY 10033, USA","{'latitude': 40.8487565, 'longitude': -73.932974}",4.3,"{'text': 'Jarto’e Sazon Restaurant', 'language...",40.848947,-73.932874,554 West 181 Street
0,"400 E 74th St, New York, NY 10021, USA","{'latitude': 40.7689029, 'longitude': -73.9547...",5.0,"{'text': 'Creative Cakes', 'languageCode': 'en'}",40.768869,-73.954483,414 East 74 Street
1,"401 E 74th St, New York, NY 10021, USA","{'latitude': 40.7689684, 'longitude': -73.9543...",NaN,"{'text': 'jared direct', 'languageCode': 'en'}",40.768869,-73.954483,414 East 74 Street
0,"280 Henry St, New York, NY 10002, USA","{'latitude': 40.7138509, 'longitude': -73.9836...",3.5,"{'text': 'Enoki Sushi', 'languageCode': 'en'}",40.713956,-73.983547,287 Henry Street
0,"345 E 194th St, Bronx, NY 10458, USA","{'latitude': 40.864104499999996, 'longitude': ...",3.9,"{'text': 'Crown Fried Chicken & Pizza', 'langu...",40.863934,-73.890993,342 East 194 Street
0,"10-12 37th Ave, Astoria, NY 11101, USA","{'latitude': 40.7598245, 'longitude': -73.9413...",4.7,{'text': 'Sunshine Coffee (fresh juices & smoo...,40.759970,-73.941429,10-14 37 Avenue
1,"9-16 37th Ave, Long Island City, NY 11101, USA","{'latitude': 40.7598303, 'longitude': -73.9413...",4.3,"{'text': 'Dominican Food', 'languageCode': 'en'}",40.759970,-73.941429,10-14 37 Avenue
0,"402 E 78th St, New York, NY 10075, USA","{'latitude': 40.7712773, 'longitude': -73.9528...",4.2,"{'text': 'Sushi of Gari UES', 'languageCode': ...",40.771451,-73.952686,409 East 78 Street
1,"401 E 78th St, New York, NY 10075, USA","{'latitude': 40.7716177, 'longitude': -73.9526...",5.0,{'text': 'Manhattan juice bar and gift and sna...,40.771451,-73.952686,409 East 78 Street


In [11]:
requested_places.head(40)

,formattedAddress,location,rating,displayName,latitude,longitude,address
0,"554 W 181st St, New York, NY 10033, USA","{'latitude': 40.8488258, 'longitude': -73.9328...",2.7,"{'text': 'Caridad', 'languageCode': 'en'}",40.848947,-73.932874,554 West 181 Street
1,"554 W 181st St, New York, NY 10033, USA","{'latitude': 40.8487565, 'longitude': -73.932974}",4.3,"{'text': 'Jarto’e Sazon Restaurant', 'language...",40.848947,-73.932874,554 West 181 Street
0,"400 E 74th St, New York, NY 10021, USA","{'latitude': 40.7689029, 'longitude': -73.9547...",5.0,"{'text': 'Creative Cakes', 'languageCode': 'en'}",40.768869,-73.954483,414 East 74 Street
1,"401 E 74th St, New York, NY 10021, USA","{'latitude': 40.7689684, 'longitude': -73.9543...",NaN,"{'text': 'jared direct', 'languageCode': 'en'}",40.768869,-73.954483,414 East 74 Street
0,"280 Henry St, New York, NY 10002, USA","{'latitude': 40.7138509, 'longitude': -73.9836...",3.5,"{'text': 'Enoki Sushi', 'languageCode': 'en'}",40.713956,-73.983547,287 Henry Street
0,"345 E 194th St, Bronx, NY 10458, USA","{'latitude': 40.864104499999996, 'longitude': ...",3.9,"{'text': 'Crown Fried Chicken & Pizza', 'langu...",40.863934,-73.890993,342 East 194 Street
0,"10-12 37th Ave, Astoria, NY 11101, USA","{'latitude': 40.7598245, 'longitude': -73.9413...",4.7,{'text': 'Sunshine Coffee (fresh juices & smoo...,40.759970,-73.941429,10-14 37 Avenue
1,"9-16 37th Ave, Long Island City, NY 11101, USA","{'latitude': 40.7598303, 'longitude': -73.9413...",4.3,"{'text': 'Dominican Food', 'languageCode': 'en'}",40.759970,-73.941429,10-14 37 Avenue
0,"402 E 78th St, New York, NY 10075, USA","{'latitude': 40.7712773, 'longitude': -73.9528...",4.2,"{'text': 'Sushi of Gari UES', 'languageCode': ...",40.771451,-73.952686,409 East 78 Street
1,"401 E 78th St, New York, NY 10075, USA","{'latitude': 40.7716177, 'longitude': -73.9526...",5.0,{'text': 'Manhattan juice bar and gift and sna...,40.771451,-73.952686,409 East 78 Street


In [12]:
df.loc[10 : 100, :]

,inspection_type,job_ticket_or_work_order_id,job_id,job_progress,bbl,boro_code,block,lot,house_number,street_name,...,location,community_board,council_district,census_tract,bin,nta,approved_date,x_coord,y_coord,address
10,BAIT,3056367,PC8577767,5,3.051220e+09,3,5122,1,1800,Albermarle Road,...,"{'latitude': '40.647009146651', 'longitude': '...",14.0,40.0,512.0,3117590.0,Flatbush,2025-12-08T15:19:59.000,994665.0,175004.0,1800 Albermarle Road
11,BAIT,3056239,PC8641500,1,2.027520e+09,2,2752,32,1175,Vyse Avenue,...,"{'latitude': '40.828009878589', 'longitude': '...",3.0,17.0,12101.0,2006173.0,Crotona Park East,NaN,NaN,NaN,1175 Vyse Avenue
12,BAIT,3056413,PC8337315,8,2.033570e+09,2,3357,32,3158,Webster Avenue,...,"{'latitude': '40.87228977812', 'longitude': '-...",7.0,11.0,425.0,2018668.0,Norwood,2025-12-09T09:20:35.000,NaN,NaN,3158 Webster Avenue
13,Compliance,14208791,PC8617662,2,1.003930e+09,1,393,18,622,East 11 Street,...,"{'latitude': '40.727064616994', 'longitude': '...",3.0,2.0,28.0,1004882.0,East Village,2025-12-09T14:58:39.000,NaN,NaN,622 East 11 Street
14,BAIT,3056394,PC8463589,9,1.021530e+09,1,2153,68,554,West 181 Street,...,"{'latitude': '40.84894689122', 'longitude': '-...",12.0,10.0,269.0,1079910.0,Washington Heights (North),2025-12-09T07:23:06.000,NaN,NaN,554 West 181 Street
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,Initial,14209494,PC8634733,1,1.019180e+09,1,1918,3,2265,Adam C Powell Blvd,...,"{'latitude': '40.814201942049', 'longitude': '...",10.0,9.0,226.0,1000000.0,Harlem (North),2025-12-16T16:27:39.000,NaN,NaN,2265 Adam C Powell Blvd
97,Initial,14208858,PC8636377,1,2.024330e+09,2,2433,70,1061,Teller Avenue,...,"{'latitude': '40.828960548053', 'longitude': '...",4.0,16.0,175.0,2002241.0,Concourse-Concourse Village,2025-12-09T13:13:35.000,NaN,NaN,1061 Teller Avenue
98,BAIT,3056387,PC8591577,4,1.011280e+09,1,1128,61,331,Columbus Avenue,...,"{'latitude': '40.779714323075', 'longitude': '...",7.0,6.0,161.0,1028789.0,Upper West Side (Central),2025-12-09T07:28:34.000,NaN,NaN,331 Columbus Avenue
99,BAIT,3056410,PC8337190,8,2.033500e+09,2,3350,25,350,East 207 Street,...,"{'latitude': '40.875052991949', 'longitude': '...",7.0,11.0,42901.0,2018447.0,Norwood,2025-12-09T09:17:41.000,NaN,NaN,350 East 207 Street


In [13]:
requested_places_saved = requested_places.copy()

requested_places_saved

,formattedAddress,location,rating,displayName,latitude,longitude,address
0,"554 W 181st St, New York, NY 10033, USA","{'latitude': 40.8488258, 'longitude': -73.9328...",2.7,"{'text': 'Caridad', 'languageCode': 'en'}",40.848947,-73.932874,554 West 181 Street
1,"554 W 181st St, New York, NY 10033, USA","{'latitude': 40.8487565, 'longitude': -73.932974}",4.3,"{'text': 'Jarto’e Sazon Restaurant', 'language...",40.848947,-73.932874,554 West 181 Street
0,"400 E 74th St, New York, NY 10021, USA","{'latitude': 40.7689029, 'longitude': -73.9547...",5.0,"{'text': 'Creative Cakes', 'languageCode': 'en'}",40.768869,-73.954483,414 East 74 Street
1,"401 E 74th St, New York, NY 10021, USA","{'latitude': 40.7689684, 'longitude': -73.9543...",NaN,"{'text': 'jared direct', 'languageCode': 'en'}",40.768869,-73.954483,414 East 74 Street
0,"280 Henry St, New York, NY 10002, USA","{'latitude': 40.7138509, 'longitude': -73.9836...",3.5,"{'text': 'Enoki Sushi', 'languageCode': 'en'}",40.713956,-73.983547,287 Henry Street
0,"345 E 194th St, Bronx, NY 10458, USA","{'latitude': 40.864104499999996, 'longitude': ...",3.9,"{'text': 'Crown Fried Chicken & Pizza', 'langu...",40.863934,-73.890993,342 East 194 Street
0,"10-12 37th Ave, Astoria, NY 11101, USA","{'latitude': 40.7598245, 'longitude': -73.9413...",4.7,{'text': 'Sunshine Coffee (fresh juices & smoo...,40.759970,-73.941429,10-14 37 Avenue
1,"9-16 37th Ave, Long Island City, NY 11101, USA","{'latitude': 40.7598303, 'longitude': -73.9413...",4.3,"{'text': 'Dominican Food', 'languageCode': 'en'}",40.759970,-73.941429,10-14 37 Avenue
0,"402 E 78th St, New York, NY 10075, USA","{'latitude': 40.7712773, 'longitude': -73.9528...",4.2,"{'text': 'Sushi of Gari UES', 'languageCode': ...",40.771451,-73.952686,409 East 78 Street
1,"401 E 78th St, New York, NY 10075, USA","{'latitude': 40.7716177, 'longitude': -73.9526...",5.0,{'text': 'Manhattan juice bar and gift and sna...,40.771451,-73.952686,409 East 78 Street


In [14]:
# import pyap

# formatted_addr = "930 2nd Ave, New York, NY 10022, USA"
# basic_addr = "930 2 Avenue"

# print(pyap.parse(formatted_addr))
# print(pyap.parse(basic_addr))

In [15]:
from fuzzywuzzy import fuzz

s1 = "930 2nd Ave, New York, NY 10022, USA"
s2 = "930 2 Avenue"

# s1 = "37 deez nuts ave"
# s2 = "38 deez nutsa ave"

print( "FuzzyWuzzy Ratio: ", fuzz.ratio(s1, s2))
print( "FuzzyWuzzy PartialRatio: ", fuzz.partial_ratio(s1, s2))
print( "FuzzyWuzzy TokenSortRatio: ", fuzz.token_sort_ratio(s1, s2))
print( "FuzzyWuzzy TokenSetRatio: ", fuzz.token_set_ratio(s1, s2))
print( "FuzzyWuzzy WRatio: ", fuzz.WRatio(s1, s2),'\n\n')


FuzzyWuzzy Ratio:  42
FuzzyWuzzy PartialRatio:  75
FuzzyWuzzy TokenSortRatio:  49
FuzzyWuzzy TokenSetRatio:  49
FuzzyWuzzy WRatio:  86 




In [19]:
requested_places_saved['ratio'] = requested_places_saved.apply(lambda x : fuzz.WRatio(x['formattedAddress'], x['address']), axis = 1)

# requested_places_saved
requested_places_saved = requested_places_saved[requested_places_saved['ratio'] >= 80]

requested_places_saved['split_check'] = requested_places_saved.apply(lambda x : x['formattedAddress'].split()[0] == x['address'].split()[0], axis = 1)

requested_places_saved = requested_places_saved[requested_places_saved['split_check'] == True]

In [20]:
requested_places_saved.to_csv("Bruh.csv", index = False)

In [2]:
from datetime import date

date.today()

datetime.date(2025, 12, 31)